In [ ]:
import pyabf
import os
import matplotlib.pyplot as plt
import scipy.signal
import numpy as np 

# Get the current working directory
print(os.getcwd())

In [ ]:
# Change the working directory
os.chdir('./data/CA1 EPSCs/ABF/12192024/EPSC')

# Print the new working directory
print(os.getcwd()) 

In [ ]:
abf = pyabf.ABF("24d19002.abf")
abf.setSweep(sweepNumber = 3, channel = 0)
print(abf.sweepY) # displays sweep data (ADC)
print(abf.sweepX) # displays sweep times (seconds)
print(abf.sweepC) # displays command waveform (DAC)

In [ ]:
plt.figure(figsize=(8, 5))
plt.ylabel(abf.sweepLabelY)
plt.xlabel(abf.sweepLabelX)
abf.setSweep(2)
plt.plot(abf.sweepX, abf.sweepY, alpha=1, color = "black", label="sweep %d" % (i))
plt.xlim(5, 25)
plt.ylim(-100, 100)
plt.show()

In [ ]:
data = abf.sweepY

# Design the filter
fs = abf.sampleRate  # Sampling rate
cutoff = 100  # Cutoff frequency in Hz
order = 4  # Filter order

b, a = scipy.signal.butter(order, cutoff / (fs / 2), btype='lowpass')

# Apply the filter
filtered_data = scipy.signal.filtfilt(b, a, data)

plt.plot(abf.sweepX, data, label='Original')
plt.plot(abf.sweepX, filtered_data, label='Filtered')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.ylabel(abf.sweepLabelY)
plt.xlabel(abf.sweepLabelX)

for i in abf.sweepList:
    abf.setSweep(i)
    filtered_data = scipy.signal.filtfilt(b, a, abf.sweepY)
    plt.plot(abf.sweepX, filtered_data, alpha=.5, label="sweep %d" % (i))
plt.show()

In [ ]:
#filtering the "artificial" events - membrane tests
data_matrix = []
threshold = -300

for i in abf.sweepList:
    abf.setSweep(i)
    num_data_pts = len(abf.sweepY)
    data = abf.sweepY
    if all(data > threshold):
        filtered_data = abf.sweepY
    else:
        filtered_data = np.zeros(num_data_pts)
        
        
    data_matrix.append(filtered_data)  

In [ ]:
plt.figure(figsize=(8, 5))
plt.ylabel(abf.sweepLabelY)
plt.xlabel(abf.sweepLabelX)
for i in abf.sweepList:
    abf.setSweep(i)
    plt.plot(abf.sweepX, data_matrix[i], alpha=.5, label="sweep %d" % (i))
plt.show()

In [ ]:
#great! now, average each of the events. 
data_average = np.mean(data_matrix, axis = 0)

plt.figure(figsize=(8, 5))
plt.ylabel(abf.sweepLabelY)
plt.xlabel(abf.sweepLabelX)
plt.plot(abf.sweepX, data_average, alpha=.5)
plt.errorbar(abf.sweepX, data_matrix)
plt.show()

sem = np.std(data_matrix, axis=0, ddof=1) / np.sqrt(data.shape[0])

plt.errorbar(abf.sweepX, sem, yerr=sem, fmt='o', capsize=5)
